In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Change project path [Anyone who is running this change to project folder]
project_path = '/content/drive/MyDrive/Colab Notebooks/CMPE259FinalProject'

%cd {project_path}

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
GradInfoLink = "https://catalog.sjsu.edu/content.php?catoid=17&navoid=7689#master"

In [ ]:
def scrape_grad_info(tag_string):
    response = requests.get(GradInfoLink)
    if response.status_code != 200:
        raise Exception("Failed to load page")

    soup = BeautifulSoup(response.content, 'html.parser')
    # find <strong>Master(s)</strong> or <strong>Doctoral</strong>
    # CHANGE THE STRING TO FIND EITHER MASTER(S) OR DOCTORAL
    grad_info_start = soup.find('strong', string=tag_string)
    if not grad_info_start:
        raise Exception("Graduate information start not found")
    # Find the <ul class="program-list"> after the <strong>Doctoral</strong>
    grad_info_section = grad_info_start.find_next('ul', class_='program-list')
    if not grad_info_section:
        raise Exception("Graduate information section not found")
    # Extract text from each <li> in the <ul>
    # Get the <a> tag inside each <li> if it exists
    # return a list containing the text and the link
    grad_info = []
    for li in grad_info_section.find_all('li'):
        a_tag = li.find('a')
        if a_tag:
            grad_info.append({
                'text': a_tag.get_text(strip=True),
                'link': "https://catalog.sjsu.edu/" + a_tag['href'] + "&print"
            })
        else:
            grad_info.append({
                'text': li.get_text(strip=True),
                'link': None
            })
    return grad_info


In [ ]:
def scrape_major_content(grad_info,file_path,string_tag):
    # for each item in grad_info, scrape the content of the link if it exists
    for item in grad_info:
        print(f"Scraping Program: {item['text']}")
        # Scrapping content from the link if it exists
        if item['link']:
            response = requests.get(item['link'])
            if response.status_code != 200:
                print(f"Failed to load page for {item['text']}")
                continue

            soup = BeautifulSoup(response.content, 'html.parser')
            # Find <td class="block_content" colspan="2">
            main_content = soup.find('td', class_='block_content', colspan='2')
            if not main_content:
                print(f"Main content not found for {item['text']}")
                continue
            # Extract text and save to .md file if there is a <li class="aclog-course"> tag keep as a single line
            # heading tags <h1>, <h2>, <h3>, <h4> should be on their own line and respect markdown syntax
            content_text = ""
            for element in main_content.descendants:
                # print(element)
                if element.name in ['h1', 'h2', 'h3', 'h4']:
                    # print(f"Found heading: {element.name} with text: {element.get_text(strip=True)}")
                    level = int(element.name[1])
                    content_text += '\n' + \
                        ('#' * level) + ' ' + element.get_text(strip=True) + '\n'
                # Handle <li> tags as single lines with new line separator '<br>'
                elif element.name == 'li':
                    # print(f"Found Bullet item: {element.get_text(strip=True)}")
                    content_text += "- " + \
                        element.get_text() + '</br>\n'
                # Grab the text only from <p> tags
                elif element.name == 'p':
                    # print(f"Found paragraph: {element.get_text(strip=True)}")
                    content_text += element.get_text() + '</br>\n'
            # Send for basic cleanup
            content_text = text_cleanup(content_text)
            # Check if item text contains '/' and replace it with '-'
            filename = string_tag + item['text'].replace('/', '-') + '.md'
            # save file to file_path
            with open(file_path + filename, 'w',encoding='utf-8') as f:
                f.write(content_text)
                print(f"Saved content to {file_path+ filename}")

In [ ]:
def text_cleanup(text):
    # Find certain phrases and delete them
    phrases_to_remove = [
        "Print this Page",
        "Facebook this Page (opens a new window)"
        "Tweet this Page (opens a new window)",
        "Return to: Academic Programs"
    ]
    for phrase in phrases_to_remove:
        text = text.replace(phrase, '')
    # find groups of lines with "-" and add a new line before and after
    lines = text.splitlines()
    cleaned_lines = []
    for line in lines:
        if line.startswith('- '):
            if cleaned_lines and cleaned_lines[-1] != '':
                cleaned_lines.append('')  # add a blank line before
            cleaned_lines.append(line)
            cleaned_lines.append('')  # add a blank line after
        else:
            cleaned_lines.append(line)
    # Join lines back
    cleaned_text = '\n'.join(cleaned_lines)
    return cleaned_text

In [ ]:
import os
# make directory
md_file_path = "CMPE259FinalContent/GradMajorReqsDocs/"
if not os.path.exists(md_file_path):
    os.makedirs(md_file_path)

# change string tag if we need doctorates
masters_grad_info = scrape_grad_info(tag_string = 'Master(s)')
print("Graduate Programs Found:")
for item in masters_grad_info:
    print(f"- {item['text']}: {item['link']}")
print("+"*40)
# change string tag if we need doctorates -> "[Doctoral] "
scrape_major_content(masters_grad_info,md_file_path,string_tag = "[Masters] ")

Graduate Programs Found:
- Accounting and Analytics, MS: https://catalog.sjsu.edu/preview_program.php?catoid=17&poid=13658&returnto=7689&print
- Aerospace Engineering, MS: https://catalog.sjsu.edu/preview_program.php?catoid=17&poid=13646&returnto=7689&print
- Applied Anthropology, MA: https://catalog.sjsu.edu/preview_program.php?catoid=17&poid=13674&returnto=7689&print
- Applied Data Intelligence, MS: https://catalog.sjsu.edu/preview_program.php?catoid=17&poid=13784&returnto=7689&print
- Applied Mathematics, MS: https://catalog.sjsu.edu/preview_program.php?catoid=17&poid=14053&returnto=7689&print
- Archives and Records Administration, MARA: https://catalog.sjsu.edu/preview_program.php?catoid=17&poid=13680&returnto=7689&print
- Art, Art History and Visual Culture Concentration, MA: https://catalog.sjsu.edu/preview_program.php?catoid=17&poid=13686&returnto=7689&print
- Art, Digital Media Art Concentration, MFA: https://catalog.sjsu.edu/preview_program.php?catoid=17&poid=13688&returnto=76